In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ==================================================
# CELL 1: Imports and Setup
# ==================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import os
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import gc
import zipfile
import random

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# === SET RANDOM SEEDS ===
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✓ Random seed set to {SEED}")


PyTorch version: 2.6.0+cu124
CUDA available: False
Using device: cpu
✓ Random seed set to 42


In [3]:
# ==================================================
# CELL 2: Extract Datasets (Same as EfficientNet)
# ==================================================

print("\n=== Locating Datasets ===")

dataset_path = "/kaggle/input/rakuten-data/"  # Adjust to your dataset name

# Auto-detect or list available
if os.path.exists(dataset_path):
    print(f"✓ Dataset found: {dataset_path}")
else:
    print("Available datasets:")
    for item in os.listdir("/kaggle/input/"):
        print(f"  /kaggle/input/{item}/")
    # Update dataset_path accordingly

# Define paths
image_train_path = os.path.join(dataset_path,"images/","image_train_preprocessed")
image_test_path = os.path.join(dataset_path,"images/", "image_test_preprocessed")

# Extract if zipped (same logic as before)
extract_base = "/kaggle/working/extracted/"

if not os.path.exists(image_train_path):
    train_zip = os.path.join(dataset_path, "image_train.zip")
    if os.path.exists(train_zip):
        print("Extracting image_train.zip...")
        os.makedirs(extract_base, exist_ok=True)
        with zipfile.ZipFile(train_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_base)
        image_train_path = os.path.join(extract_base, "image_train")

if not os.path.exists(image_test_path):
    test_zip = os.path.join(dataset_path, "image_test.zip")
    if os.path.exists(test_zip):
        print("Extracting image_test.zip...")
        os.makedirs(extract_base, exist_ok=True)
        with zipfile.ZipFile(test_zip, 'r') as zip_ref:
            zip_ref.extractall(extract_base)
        image_test_path = os.path.join(extract_base, "image_test")

assert os.path.exists(image_train_path), f"Train path not found: {image_train_path}"
assert os.path.exists(image_test_path), f"Test path not found: {image_test_path}"

print(f"✓ Train path: {image_train_path}")
print(f"✓ Test path: {image_test_path}")



=== Locating Datasets ===
✓ Dataset found: /kaggle/input/rakuten-data/
✓ Train path: /kaggle/input/rakuten-data/images/image_train_preprocessed
✓ Test path: /kaggle/input/rakuten-data/images/image_test_preprocessed


In [4]:
# ==================================================
# CELL 3: Configuration and Transforms
# ==================================================

BATCH_SIZE = 128  # Larger for Kaggle GPU
IMG_SIZE = 224
NUM_WORKERS = 2

print(f"\n=== Configuration ===")
print(f"Batch size: {BATCH_SIZE}")
print(f"Image size: {IMG_SIZE}")
print(f"Seed: {SEED}")

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), value='random')
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class TransformDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y
    
    def __len__(self):
        return len(self.subset)


=== Configuration ===
Batch size: 128
Image size: 224
Seed: 42


In [5]:
# ==================================================
# CELL 4: Load and Split Datasets
# ==================================================

print("\n=== Loading Datasets ===")

train_full_dataset = datasets.ImageFolder(root=image_train_path, transform=None)
num_classes = len(train_full_dataset.classes)
class_names = train_full_dataset.classes

print(f"Training images: {len(train_full_dataset)}")
print(f"Classes: {num_classes}")

test_full_dataset = datasets.ImageFolder(root=image_test_path, transform=None)
print(f"Test images: {len(test_full_dataset)}")

assert test_full_dataset.classes == class_names, "Class mismatch!"

# Split with SAME SEED as EfficientNet
train_size = int(0.8 * len(train_full_dataset))
val_size = len(train_full_dataset) - train_size

train_subset, val_subset = random_split(
    train_full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)  # CRITICAL: Same seed
)

print(f"\nSplits: Train={len(train_subset)}, Val={len(val_subset)}, Test={len(test_full_dataset)}")

train_dataset = TransformDataset(train_subset, transform=train_transforms)
val_dataset = TransformDataset(val_subset, transform=val_transforms)
test_dataset = TransformDataset(test_full_dataset, transform=val_transforms)


=== Loading Datasets ===
Training images: 65394
Classes: 27
Test images: 16737

Splits: Train=52315, Val=13079, Test=16737


In [6]:
# ==================================================
# CELL 5: Create DataLoaders
# ==================================================

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Batches - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")


Batches - Train: 409, Val: 103, Test: 131


In [7]:
# ==================================================
# CELL 6: Define Basic CNN Model
# ==================================================

class BasicCNN(nn.Module):
    def __init__(self, num_classes, dropout=0.5):
        super(BasicCNN, self).__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        self.block4 = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.global_avg_pool(x)
        x = self.classifier(x)
        return x

model = BasicCNN(num_classes=num_classes, dropout=0.5).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\nBasic CNN Parameters: {total_params:,}")


Basic CNN Parameters: 2,467,163


In [8]:
# ==================================================
# CELL 7: Training Functions
# ==================================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': loss.item(), 'acc': 100 * correct / total})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [9]:
# ==================================================
# CELL 8: Training Loop
# ==================================================

print("\n" + "="*60)
print("TRAINING BASIC CNN FROM SCRATCH")
print("="*60)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.2, patience=7, min_lr=1e-7
)

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15
EPOCHS = 45
execute_train = 0

if execute_train == 1:
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS} - LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = validate_epoch(model, val_loader, criterion, device)
        
        print(f"Train: {train_acc:.2f}%, Val: {val_acc:.2f}%")
        
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_basic_cnn.pth')
            print(f"✓ Saved! Best: {val_acc:.2f}%")
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

gc.collect()
torch.cuda.empty_cache()



TRAINING BASIC CNN FROM SCRATCH


In [10]:
# ==================================================
# CELL 8.5: Load the Best Model (Alternative to 8)
# ==================================================

# Path will be: /kaggle/input/your-dataset-slug/
MODEL_PATH = "/kaggle/input/best-basic-cnn/best_basic_cnn.pth"

# Check if file exists
if os.path.exists(MODEL_PATH):
    print(f"✓ Model found: {MODEL_PATH}")
else:
    print("Available files:")
    for root, dirs, files in os.walk("/kaggle/input/"):
        for file in files:
            print(f"  {os.path.join(root, file)}")

# Load model weights
#model = BasicCNN(num_classes=27)  # Define architecture first
#model.load_state_dict(torch.load(MODEL_PATH))
#model.eval()

#print("✓ Model loaded successfully!")

✓ Model found: /kaggle/input/best-basic-cnn/best_basic_cnn.pth


In [11]:
# ==================================================
# CELL 9: Final Evaluation
# ==================================================

print("\n" + "="*60)
print("FINAL EVALUATION")
print("="*60)

if execute_train == 1:
    model.load_state_dict(torch.load('best_basic_cnn.pth',map_location=device))
else:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

#_, train_final_acc = train_epoch(model, train_loader, criterion, optimizer, device)
_, val_final_acc, y_pred_val, y_true_val = validate_epoch(model, val_loader, criterion, device)
_, test_acc, y_pred_test, y_true_test = validate_epoch(model, test_loader, criterion, device)

print(f"\n{'='*60}")
print("BASIC CNN RESULTS")
print(f"{'='*60}")
#print(f"Train: {train_final_acc:.2f}%")
print(f"Validation: {val_final_acc:.2f}%")
print(f"TEST:       {test_acc:.2f}%")
print(f"{'='*60}")

with open('basic_cnn_results.txt', 'w') as f:
#    f.write(f"Train: {train_final_acc:.2f}%")
    f.write(f"Validation: {val_final_acc:.2f}%")
    f.write(f"TEST:       {test_acc:.2f}%")
    
report_test = classification_report(y_true_test, y_pred_test, target_names=class_names)
print("\n" + report_test)

with open('classification_report_BasicCNN_TEST.txt', 'w') as f:
    f.write(f"Test: {test_acc:.2f}%, Val: {val_final_acc:.2f}%\n\n")
    f.write(report_test)

cm_test = confusion_matrix(y_true_test, y_pred_test)
np.save('confusion_matrix_BasicCNN_TEST.npy', cm_test)

print("\n✓ Basic CNN training complete!")


FINAL EVALUATION


Validation: 100%|██████████| 131/131 [1:13:05<00:00, 33.47s/it, loss=1.7, acc=52.2]


BASIC CNN RESULTS
Validation: 51.90%
TEST:       52.15%

              precision    recall  f1-score   support

          10       0.55      0.50      0.52       623
        1140       0.50      0.50      0.50       534
        1160       0.68      0.93      0.78       791
        1180       0.69      0.06      0.11       153
        1280       0.32      0.43      0.37       971
        1281       0.30      0.14      0.19       413
        1300       0.39      0.75      0.51       996
        1301       0.81      0.11      0.19       159
        1302       0.38      0.09      0.14       497
        1320       0.46      0.21      0.29       645
        1560       0.39      0.54      0.45      1002
        1920       0.68      0.71      0.69       860
        1940       0.62      0.38      0.47       160
        2060       0.47      0.35      0.40       998
        2220       0.45      0.10      0.17       165
        2280       0.67      0.76      0.71       952
        2403       0.58

In [12]:
# ==================================================
# CELL 10: Download Results
# ==================================================

from IPython.display import FileLink

print("\n=== Download Files ===")
display(FileLink('best_basic_cnn.pth'))
display(FileLink('basic_cnn_results.txt'))
display(FileLink('classification_report_BasicCNN_TEST.txt'))
display(FileLink('confusion_matrix_BasicCNN_TEST.npy'))


=== Download Files ===


/kaggle/working/best_basic_cnn.pth

/kaggle/working/basic_cnn_results.txt

/kaggle/working/classification_report_BasicCNN_TEST.txt

/kaggle/working/confusion_matrix_BasicCNN_TEST.npy